In [15]:
import folium
import json
import requests

In [18]:


# 1. Инициализируем карту
m = folium.Map(location=[55.609857, 37.534991], zoom_start=12, tiles="CartoDB positron")

# ВСТАВЬТЕ СЮДА ВАШ API-КЛЮЧ ИЗ КАБИНЕТА РАЗРАБОТЧИКА
YANDEX_API_KEY = "bfea0fa5-2a5f-4af0-bdf8-ce939cdc430b"

# 2. Добавляем полигоны районов ЮЗАО
folium.GeoJson(
    "moscow_districts.geojson",
    name="Районы ЮЗАО",
    style_function=lambda x: {
        'fillColor': "#828282",
        'fillOpacity': 0.15,
        'color': "#000000",
        'weight': 0.7
    },
    tooltip=folium.GeoJsonTooltip(fields=['district'], aliases=["Район:"], localize=True),
    interactive=True
).add_to(m)

# 3. Собираем координаты из ОБОИХ файлов
raw_coordinates = []

with open("points.geojson", "r", encoding="utf-8") as f:
    data1 = json.load(f)
    for feature in data1['features']:
        raw_coordinates.append(feature['geometry']['coordinates'])

with open("station of monitoring.geojson", "r", encoding="utf-8") as f:
    data2 = json.load(f)
    for feature in data2['features']:
        raw_coordinates.append(feature['geometry']['coordinates'])

# --- АЛГОРИТМ СОРТИРОВКИ «БЛИЖАЙШИЙ СОСЕД» ---
def sort_by_distance(coords):
    if not coords:
        return []
    unvisited = coords.copy()
    sorted_coords = [unvisited.pop(0)]
    while unvisited:
        current = sorted_coords[-1]
        closest = min(unvisited, key=lambda c: math.hypot(c[0] - current[0], c[1] - current[1]))
        sorted_coords.append(closest)
        unvisited.remove(closest)
    return sorted_coords

# Упорядочиваем точки
all_coordinates = sort_by_distance(raw_coordinates)


# 4. ЗАПРОС МАРШРУТА НАПРЯМУЮ К ЯНДЕКСУ
# Формируем строку координат для Яндекса в формате "долгота,широта|долгота,широта"
points_string = "|".join([f"{c[0]},{c[1]}" for c in all_coordinates])

url = f"https://api.routing.yandex.net/v2/route?waypoints={points_string}&mode=driving&apikey={YANDEX_API_KEY}"

try:
    response = requests.get(url)
    route_data = response.json()
    
    # Извлекаем все точки получившейся геометрии маршрута
    route_coords = []
    for route_option in route_data['route']['features']:
        if route_option['geometry']['type'] == 'LineString':
            # Яндекс возвращает [широта, долгота], инверсия не нужна
            for coord in route_option['geometry']['coordinates']:
                route_coords.append([coord[0], coord[1]])

    # 5. Отрисовка линии маршрута Яндекса
    folium.PolyLine(
        locations=route_coords,
        color="#4CAF50",  # Красивый зеленый трек
        weight=5,
        opacity=0.8,
        name="Маршрут (Яндекс)"
    ).add_to(m)
    
except Exception as e:
    print(f"Ошибка при получении маршрута от Яндекса: {e}")
    print("Проверьте правильность введённого API-ключа.")


# 6. ОТРИСОВЫВАЕМ КАСТОМНЫЕ МАРКЕРЫ-КАПЛИ

# Слой 1: Наши точки (Черные капли)
folium.GeoJson(
    "points.geojson", 
    name="Наши точки",
    marker=folium.Marker(
        icon=folium.DivIcon(
            html="""
            <div style="position: relative; transform: translate(-50%, -100%); width: 16px; height: 22px;">
                <div style="position: absolute; width: 16px; height: 16px; background-color: #212121; border-radius: 50% 50% 50% 0; transform: rotate(-45deg); box-shadow: -1px 2px 4px rgba(0,0,0,0.3);"></div>
                <div style="position: absolute; top: 5px; left: 5px; width: 6px; height: 6px; background-color: #ffffff; border-radius: 50%;"></div>
            </div>
            """
        )
    ),
    tooltip=folium.GeoJsonTooltip(fields=['title'], aliases=["Объект: "], localize=True)
).add_to(m)

# Слой 2: Станции мониторинга (Красные капли)
folium.GeoJson(
    "station of monitoring.geojson", 
    name="Станции мониторинга",
    marker=folium.Marker(
        icon=folium.DivIcon(
            html="""
            <div style="position: relative; transform: translate(-50%, -100%); width: 16px; height: 22px;">
                <div style="position: absolute; width: 16px; height: 16px; background-color: #FF0000; border-radius: 50% 50% 50% 0; transform: rotate(-45deg); box-shadow: -1px 2px 4px rgba(0,0,0,0.3);"></div>
                <div style="position: absolute; top: 5px; left: 5px; width: 6px; height: 6px; background-color: #ffffff; border-radius: 50%;"></div>
            </div>
            """
        )
    ),
    tooltip=folium.GeoJsonTooltip(fields=['description'], aliases=["Станция: "], localize=True)
).add_to(m)

# 7. Панель управления слоями
folium.LayerControl().add_to(m)

# Сохраняем итоговый результат
m.save("combined_route_map.html")
print("Карта успешно создана!")

Ошибка при получении маршрута от Яндекса: 'route'
Проверьте правильность введённого API-ключа.
Карта успешно создана!
